## **Cleaning and EDA**

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import re
from build_dataset import DATA_DIR, FILE_ORDER

TARGET = "Label"

1. **Merge + Initial Filtering of Files**

The CIC-IDS2017 dataset captures network traffic data over a week (July 3-7 2017). Data is split into 8 csv files based on day of week and attack type, but was captured continuously with the same methodology, making merging valid.

In [2]:
# VERIFY SCHEMAS OF INDIVIDUAL DATA FILES
schemas = {}
for file in FILE_ORDER:
    cols = pd.read_csv(f"../data/{file}", nrows=0).columns.str.strip().tolist()
    cols = [c for c in cols if c != "Fwd Header Length.1"] # Normalize repeated header
    schemas[file] = tuple(cols)

unique_schemas = set(schemas.values())
print(f"{len(unique_schemas)} distinct schema(s) across data files.")


1 distinct schema(s) across data files.


In [3]:
# FILTER INDIVIDUAL DATA FILES AND MERGE
writer = None
schema_ref = None
MERGED_PATH = os.path.join(DATA_DIR, "merged_rawlabels.parquet")

for file in FILE_ORDER:
    path = os.path.join(DATA_DIR, file)
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()

    # Fix non-UTF8 encoding
    df[TARGET] = df[TARGET].astype(str).str.strip().str.replace("\ufffd", "-", regex=False)

    # Drop rows with missing and infinite values
    # TO-DO: REVISIT VALIDITY OF DROPPING SUCH ROWS
    feat_cols = [c for c in df.columns if c != TARGET]
    valid = np.ones(len(df), dtype=bool)
    for c in feat_cols:
        v = df[c].to_numpy()
        if np.issubdtype(v.dtype, np.number):
            valid &= np.isfinite(v.astype(np.float64, copy=False))
    df = df.loc[valid]

    # Drop within file duplicates
    # TO-DO: REVISIT VALIDITY OF DROPPING DUPLICATES
    df = df.drop_duplicates()

    # Reduce memory usage by using float32
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].astype(np.float32)

    # Display length of each source file
    df["Source File"] = file
    print(f"{file}: {len(df):,} rows")

    # Convert to Pyarrow Table and write into file
    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None: # First file only
        schema_ref = table.schema
        writer = pq.ParquetWriter(MERGED_PATH, schema_ref)
    else:
        table = table.cast(schema_ref)
    writer.write_table(table)
    del df, table

writer.close()
print("\nMerged to path:", MERGED_PATH)

Monday-WorkingHours.pcap_ISCX.csv: 502,650 rows
Tuesday-WorkingHours.pcap_ISCX.csv: 421,626 rows
Wednesday-workingHours.pcap_ISCX.csv: 610,492 rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 164,179 rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 252,790 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv: 184,044 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 213,777 rows
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 223,082 rows

Merged to path: c:\Users\lixin\portfolio\cybersecurity_project\processing\..\src\..\data\merged_rawlabels.parquet


In [4]:
# FILTERED MERGED DATA
pf = pq.ParquetFile(MERGED_PATH)
FINAL_PATH = os.path.join(DATA_DIR, "merge_complete.parquet")
all_cols = list(pf.schema_arrow.names)
feat_cols = [c for c in all_cols if c not in (TARGET, "source_file")]

# Find inter-file duplicates and constant columns
seen = set()
dup_flags = []
col_uniques = {c: set() for c in feat_cols}
for batch in pf.iter_batches(batch_size=250_000): # Work on data in batches
    # Checks for duplicates and inspects column values
    chunk = batch.to_pandas()
    h = pd.util.hash_pandas_object(chunk[feat_cols + [TARGET]], index=False).to_numpy()
    flags = np.array([(v in seen) or seen.add(v) for v in h], dtype=bool) # Checks if hash was seen
    dup_flags.append(flags)
    
    #Track unique values to find constant columns
    for c in feat_cols:
        if len(col_uniques[c]) <= 1:
            col_uniques[c].update(pd.unique(chunk[c])[:3].tolist())
dup_mask = np.concatenate(dup_flags)
constant_cols = [c for c, u in col_uniques.items() if len(u) <= 1]
keep_cols = [c for c in all_cols if c not in constant_cols] # Filters out constant columns
print(f"Number of inter-file duplicates: {dup_mask.sum()}")
print(f"Number of constant columns: {len(constant_cols)}")

# Write filtered file
writer = None
row_ptr = 0
rows_out = 0

# Restream file loading only kept columns
for batch in pq.ParquetFile(MERGED_PATH).iter_batches(batch_size=250_000, columns=keep_cols):
    chunk = batch.to_pandas()
    m = ~dup_mask[row_ptr: row_ptr + len(chunk)] # Boolean: Duplicate mask
    row_ptr += len(chunk)
    out = chunk.loc[m] # Keep only unique rows
    rows_out += len(out)

    # Write file
    table = pa.Table.from_pandas(out, preserve_index=False) # Convert to Pyarrow Table
    if writer is None:
        writer = pq.ParquetWriter(FINAL_PATH, table.schema)
    writer.write_table(table) 
    
writer.close()
print(f"\nWritten to path:", FINAL_PATH)

Number of inter-file duplicates: 205
Number of constant columns: 8

Written to path: c:\Users\lixin\portfolio\cybersecurity_project\processing\..\src\..\data\merge_complete.parquet


In [ ]:

def to_camel_case(col):
    # Split on any non-alphanumeric character
    parts = re.split(r"[^0-9a-zA-Z]+", col.strip())
    parts = [p for p in parts if p]  # remove empty strings
    return "".join(p.capitalize() for p in parts)

# Use function to convert all column names to CamelCase
table = pq.read_table(FINAL_PATH) 
new_schema = pa.schema([ 
    pa.field(to_camel_case(name), table.schema.field(name).type)
    for name in table.schema.names
])
new_table = table.rename_columns([to_camel_case(c) for c in table.schema.names])
print("New column names:", new_table.schema.names)

# Overwrite with column name changes
pq.write_table(new_table, FINAL_PATH)
print("Column names converted to CamelCase in:", FINAL_PATH)
